# 📊 Notebook 5 — Dashboard & Visualisation Report
**Input:** `final_dataset.csv` + `outputs/*.json`

This notebook reproduces all dashboard charts inline — no Streamlit server needed.
It covers:
1. Key KPIs and acceptance overview
2. Time-of-day deep dives
3. Topic category analysis
4. RobBERT sentiment & tone trends
5. Model performance comparison
6. How to launch the interactive Streamlit dashboard


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import json, os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'figure.facecolor': 'white',
})
BLUE, RED, GREEN, ORANGE = '#2B5797','#C0392B','#27AE60','#E67E22'
GREY = '#95A5A6'
TONE_COLORS = {'aggressive':'#E74C3C','mean':'#E67E22','neutral':'#95A5A6',
               'peaceful':'#3498DB','kind':'#27AE60','happy':'#F1C40F'}

df = pd.read_csv("final_dataset.csv")
df['Topic_date'] = pd.to_datetime(df['Topic_date'], errors='coerce')
HAS_BERT = 'sentiment_label' in df.columns

print(f"Loaded {len(df)} rows | BERT columns available: {HAS_BERT}")


## 1. Key Performance Indicators

In [ ]:
total   = len(df)
acc     = (df['Decision']=='Accepted').sum()
rej     = (df['Decision']=='Rejected').sum()
acc_pct = acc/total*100
plenary_acc = df[df['Meeting_type']=='Plenair']['label'].mean()*100
comm_acc    = df[df['Meeting_type']=='Commissie']['label'].mean()*100

print("=" * 50)
print(f"  Total Motions:           {total:,}")
print(f"  Accepted:                {acc:,}  ({acc_pct:.1f}%)")
print(f"  Rejected:                {rej:,}  ({100-acc_pct:.1f}%)")
print(f"  Plenary Acceptance Rate: {plenary_acc:.1f}%")
print(f"  Committee Acceptance:    {comm_acc:.1f}%")
print(f"  Date range:              {df['Topic_date'].min().date()} → {df['Topic_date'].max().date()}")
print("=" * 50)


## 2. Time-of-Day Deep Dive

In [ ]:
TOD_ORDER = ['Early morning','Late morning','Early afternoon','Late afternoon','Evening','Night']

tod = (df.groupby('Time_of_day_category')['label']
         .agg(['mean','count','sum'])
         .reindex(TOD_ORDER)
         .rename(columns={'mean':'acc','count':'total','sum':'accepted'}))
tod['rejected'] = tod['total'] - tod['accepted']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Time-of-Day Analysis', fontsize=15, fontweight='bold', y=1.01)

# A: Acceptance rate
ax = axes[0,0]
bar_colors = [GREEN if v>=50 else RED for v in tod['acc']*100]
bars = ax.bar(tod.index, tod['acc']*100, color=bar_colors, edgecolor='white', alpha=0.85, width=0.6)
ax.axhline(50, color='grey', linestyle='--', lw=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Acceptance Rate by Time of Day', fontweight='bold')
ax.set_ylim(0, 72)
for bar, n in zip(bars, tod['total']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'n={n}', ha='center', va='bottom', fontsize=8, color='#555')
ax.tick_params(axis='x', rotation=25)

# B: Volume stacked bar
ax2 = axes[0,1]
x = range(len(TOD_ORDER))
ax2.bar(x, tod['accepted'], label='Accepted', color=GREEN, edgecolor='white', alpha=0.85)
ax2.bar(x, tod['rejected'], bottom=tod['accepted'], label='Rejected', color=RED, edgecolor='white', alpha=0.85)
ax2.set_xticks(list(x)); ax2.set_xticklabels(TOD_ORDER, rotation=25, ha='right')
ax2.set_title('Volume by Time of Day (stacked)', fontweight='bold')
ax2.legend(); ax2.set_ylabel('Count')

# C: Hour histogram
ax3 = axes[1,0]
df_h = df[df['hour']>=0]
for decision, color in [('Accepted', GREEN), ('Rejected', RED)]:
    ax3.hist(df_h[df_h['Decision']==decision]['hour'], bins=24, range=(0,24),
             alpha=0.6, color=color, label=decision, edgecolor='white')
ax3.set_title('Session Volume by Start Hour', fontweight='bold')
ax3.set_xlabel('Hour of Day'); ax3.set_ylabel('Count')
ax3.legend(); ax3.set_xticks(range(0,25,2))

# D: Season × Meeting type
ax4 = axes[1,1]
sm = (df.groupby(['Season','Meeting_type'])['label']
        .mean().unstack()*100)
season_order = ['Spring','Summer','Autumn','Winter']
sm = sm.reindex(season_order)
x4 = np.arange(len(season_order))
w = 0.35
for i, (mt, color) in enumerate([('Plenair', BLUE), ('Commissie', ORANGE)]):
    if mt in sm.columns:
        ax4.bar(x4+i*w, sm[mt], w, label=mt, color=color, alpha=0.85, edgecolor='white')
ax4.set_xticks(x4+w/2); ax4.set_xticklabels(season_order)
ax4.yaxis.set_major_formatter(mtick.PercentFormatter())
ax4.axhline(50, color='grey', linestyle='--', lw=0.8)
ax4.set_title('Acceptance by Season × Meeting Type', fontweight='bold')
ax4.legend(); ax4.set_ylim(0, 75)

plt.tight_layout(); plt.show()


In [ ]:
# ── Hour × Day-of-week heatmap ────────────────────────────────────────────
df_h = df[(df['hour']>=0) & df['day_of_week'].notna()]
hm = df_h.groupby(['hour','day_of_week'])['label'].mean().unstack(fill_value=np.nan)
hm.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][:len(hm.columns)]

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(hm, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.3, linecolor='white', annot=True, fmt='.2f',
            cbar_kws={'label':'Acceptance Rate', 'format':'%.0%%'})
ax.set_title('Acceptance Rate: Hour of Day × Day of Week', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Topic Category Analysis

In [ ]:
cat = (df.groupby('Topic_category')['label']
         .agg(['mean','count'])
         .sort_values('mean')
         .rename(columns={'mean':'acc','count':'n'}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Acceptance rate
ax = axes[0]
colors_c = [GREEN if v > 0.5 else RED for v in cat['acc']]
h = ax.barh(cat.index, cat['acc']*100, color=colors_c, edgecolor='white', alpha=0.85)
ax.axvline(50, color='grey', linestyle='--', lw=0.8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Acceptance Rate by Topic Category', fontweight='bold')
for bar, n in zip(h, cat['n']):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'  n={n}', va='center', fontsize=8)

# Scatter: volume vs acceptance
ax2 = axes[1]
sc = ax2.scatter(cat['n'], cat['acc']*100,
                  c=cat['acc'], cmap='RdYlGn', vmin=0.4, vmax=0.7,
                  s=150, edgecolors='white', linewidth=1.2, zorder=3)
for cat_name, row in cat.iterrows():
    ax2.annotate(cat_name.replace(' & ','
& '),
                  (row['n'], row['acc']*100),
                  textcoords='offset points', xytext=(6, 0),
                  fontsize=7, va='center')
ax2.axhline(50, color='grey', linestyle='--', lw=0.8)
ax2.set_xlabel('Number of Motions'); ax2.set_ylabel('Acceptance Rate (%)')
ax2.set_title('Volume vs Acceptance Rate', fontweight='bold')
plt.colorbar(sc, ax=ax2, label='Acceptance Rate')
plt.tight_layout(); plt.show()


## 4. RobBERT Sentiment & Tone Trends

In [ ]:
if not HAS_BERT:
    print("⚠️  BERT columns not found — run Notebook 3 first to generate final_dataset.csv")
else:
    # Distribution overview
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('RobBERT Sentiment & Tone Overview', fontsize=13, fontweight='bold')

    # Sentiment pie
    counts = df['sentiment_label'].value_counts()
    axes[0].pie(counts, labels=counts.index,
                colors=[GREEN if l=='positive' else RED for l in counts.index],
                autopct='%1.1f%%', startangle=90,
                wedgeprops={'edgecolor':'white','linewidth':1.5})
    axes[0].set_title('Sentiment Distribution')

    # Tone bar
    tone_counts = df['tone_label'].value_counts()
    axes[1].bar(tone_counts.index, tone_counts.values,
                color=[TONE_COLORS.get(t, GREY) for t in tone_counts.index],
                edgecolor='white', alpha=0.9)
    axes[1].set_title('Tone Distribution')
    axes[1].tick_params(axis='x', rotation=25)

    # Sentiment vs acceptance
    acc_s = df.groupby('sentiment_label')['label'].mean()*100
    axes[2].bar(acc_s.index, acc_s.values,
                color=[GREEN if l=='positive' else RED for l in acc_s.index],
                edgecolor='white', alpha=0.85)
    axes[2].axhline(50, color='grey', linestyle='--')
    axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Acceptance Rate by Sentiment')
    axes[2].set_ylim(0,70)
    plt.tight_layout(); plt.show()


In [ ]:
if HAS_BERT:
    # Tone × Category heatmap
    pivot = (df.groupby(['Topic_category','tone_label'])['Id']
               .count().unstack(fill_value=0))
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)*100

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(pivot_pct, ax=ax, cmap='YlOrRd', linewidths=0.4,
                linecolor='white', annot=True, fmt='.0f',
                cbar_kws={'label':'% of category'})
    ax.set_title('Tone Distribution by Topic Category (%)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()


In [ ]:
if HAS_BERT:
    # Sentiment over time
    df_ts = df.dropna(subset=['Topic_date','sentiment_score']).copy()
    df_ts = df_ts.set_index('Topic_date')
    q_mean = df_ts['sentiment_score'].resample('QE').mean()
    q_std  = df_ts['sentiment_score'].resample('QE').std()

    fig, ax = plt.subplots(figsize=(13, 3.5))
    ax.plot(q_mean.index, q_mean.values, color=BLUE, lw=2, label='Mean sentiment')
    ax.fill_between(q_mean.index,
                     q_mean.values - q_std.values,
                     q_mean.values + q_std.values,
                     alpha=0.15, color=BLUE, label='±1 std')
    ax.fill_between(q_mean.index, q_mean.values, 0,
                     where=(q_mean.values>0), alpha=0.2, color=GREEN)
    ax.fill_between(q_mean.index, q_mean.values, 0,
                     where=(q_mean.values<=0), alpha=0.2, color=RED)
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_title('RobBERT Sentiment Score over Time (quarterly)', fontweight='bold')
    ax.set_xlabel('Date'); ax.set_ylabel('Sentiment Score')
    ax.legend()
    plt.tight_layout(); plt.show()


## 5. Model Performance Comparison

In [ ]:
results_dir = "outputs"
model_results = {}
if os.path.isdir(results_dir):
    for fname in os.listdir(results_dir):
        if fname.endswith('_results.json'):
            with open(os.path.join(results_dir, fname)) as f:
                r = json.load(f)
                model_results[r['model']] = r

if model_results:
    comp = pd.DataFrame([{
        'Model':        r['model'],
        'Accuracy':     r.get('accuracy', np.nan),
        'F1':           r.get('f1', np.nan),
        'ROC-AUC':      r.get('roc_auc', np.nan),
        'Brier Score↓': r.get('brier', np.nan),
    } for r in model_results.values()]).sort_values('ROC-AUC', ascending=False)

    print(comp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # Bar chart
    metrics = ['Accuracy','F1','ROC-AUC']
    x = np.arange(len(metrics))
    w = 0.22
    colors_m = [BLUE, ORANGE, GREEN, RED]

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (_, row) in enumerate(comp.iterrows()):
        vals = [row['Accuracy'], row['F1'], row['ROC-AUC']]
        bars = ax.bar(x+i*w, vals, w, label=row['Model'],
                      color=colors_m[i%len(colors_m)], alpha=0.85, edgecolor='white')
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                    f"{bar.get_height():.3f}", ha='center', fontsize=7.5)
    ax.set_xticks(x+w); ax.set_xticklabels(metrics, fontsize=11)
    ax.set_ylim(0, 1.08); ax.set_ylabel('Score')
    ax.set_title('Model Comparison — Test Set', fontsize=13, fontweight='bold')
    ax.axhline(0.5, color='grey', linestyle=':', lw=0.8)
    ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("⚠️  No model results found. Run Notebook 4 first.")


## 6. Launch Streamlit Dashboard

To launch the interactive dashboard with full filter controls, run this from your terminal:

```bash
pip install streamlit plotly
streamlit run dashboard_app.py
```

Or run the cell below to start it from the notebook:


In [ ]:
# ── Inline mini-dashboard (static summary, no server needed) ────────────────
print("\n🏛  DUTCH PARLIAMENT MOTION PIPELINE — SUMMARY REPORT")
print("=" * 60)

total = len(df)
acc_r = (df['label']==1).mean()*100
print(f"  Total motions analysed:   {total:,}")
print(f"  Overall acceptance rate:  {acc_r:.1f}%")

tod_top = df.groupby('Time_of_day_category')['label'].mean().idxmax()
tod_bot = df.groupby('Time_of_day_category')['label'].mean().idxmin()
print(f"  Best time to vote:        {tod_top}")
print(f"  Worst time to vote:       {tod_bot}")

cat_top = df.groupby('Topic_category')['label'].mean().idxmax()
print(f"  Highest-acceptance topic: {cat_top}")

if model_results:
    best = max(model_results.values(), key=lambda r: r.get('roc_auc',0))
    print(f"  Best predictive model:    {best['model']} (AUC={best['roc_auc']:.4f})")

print("=" * 60)
print("\nTo launch Streamlit dashboard:")
print("  streamlit run dashboard_app.py -- --data final_dataset.csv")
